<a href="https://colab.research.google.com/github/Vamsiyadav1472/DSA0302/blob/main/DSA03_Natural_Language_Processing_Assignment_1_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Code 1: Regular Expression Tokenizer & Normalizer (Task 3)

In [8]:
import re

def clinical_tokenizer(text):
    # Wrapped the regex in double quotes r"..." to allow literal single quotes (') inside the expression without syntax errors
    token_pattern = r"\b\d+(?:\.\d+)?(?:mg|ml|kg|cm|mm)?\b|[A-Za-z]+(?:'[-A-Za-z]+)*|\b[A-Za-z]\.\b|[0-9]{1,2}/[0-9]{1,2}/[0-9]{2,4}"
    tokens = re.findall(token_pattern, text)
    return tokens

def clinical_normalizer(tokens):
    abbreviations = {
        "bp": "blood pressure",
        "temp": "temperature",
        "bid": "twice daily",
        "tid": "three times daily",
        "hx": "history"
    }
    normalized = []
    for token in tokens:
        lower_token = token.lower()
        if lower_token in abbreviations:
            normalized.append(abbreviations[lower_token])
        else:
            normalized.append(lower_token)
    return normalized

# Example usage:
sample_text = "Patient BP 120/80, temp 98.6F, prescribed tab. paracetamol 500mg tid."
tokens = clinical_tokenizer(sample_text)
normalized_tokens = clinical_normalizer(tokens)
print("Tokens:", tokens)
print("Normalized:", normalized_tokens)

Tokens: ['Patient', 'BP', '120', '80', 'temp', '98', 'F', 'prescribed', 'tab', 'paracetamol', '500mg', 'tid']
Normalized: ['patient', 'blood pressure', '120', '80', 'temperature', '98', 'f', 'prescribed', 'tab', 'paracetamol', '500mg', 'three times daily']


Code 2: Structured Entity Recognition Patterns (Task 2)

In [5]:
import re

CLINICAL_PATTERNS = {
    "patient_id": r'\bPT-[0-9]{5}\b',
    "admission_date": r'\b[0-9]{4}-[0-9]{2}-[0-9]{2}\b',
    "drug_with_strength": r'\b[A-Za-z]+\s+[0-9]+(?:mg|ml|g)\b',
    "dosage_frequency": r'\b(?:OD|BD|TDS|QDS|STAT)\b',
    "temperature": r'\b[0-9]{2,3}(?:\.[0-9])?\s*(?:°C|°F|F|C)\b',
    "icd_code": r'\b[A-Z][0-9]{2}(?:\.[0-9]{1,2})?\b'
}

def extract_entities(text):
    extracted = {}
    for entity_type, pattern in CLINICAL_PATTERNS.items():
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            extracted[entity_type] = matches
    return extracted

# Example usage:
record_text = "Patient PT-12345 admitted on 2026-03-01. Diagnosis ICD J45.9. Administer paracetamol 500mg BD. Temp 101.2F."
print(extract_entities(record_text))

{'patient_id': ['PT-12345'], 'admission_date': ['2026-03-01'], 'drug_with_strength': ['paracetamol 500mg'], 'dosage_frequency': ['BD'], 'temperature': ['101.2F'], 'icd_code': ['J45.9']}


Code 3: Finite State Automaton Validator (Task 4)

In [4]:
class PatientIDValidatorDFA:
    def __init__(self):
        self.state = 'S0'

    def transition(self, char):
        if self.state == 'S0':
            return 'S1' if char == 'P' else 'TRAP'
        elif self.state == 'S1':
            return 'S2' if char == 'T' else 'TRAP'
        elif self.state == 'S2':
            return 'S3' if char == '-' else 'TRAP'
        elif self.state in ['S3', 'S4', 'S5', 'S6', 'S7']:
            return f'S{int(self.state[1]) + 1}' if char.isdigit() else 'TRAP'
        return 'TRAP'

    def validate(self, input_string):
        self.state = 'S0'
        for char in input_string:
            self.state = self.transition(char)
            if self.state == 'TRAP':
                return False
        return self.state == 'S8'

# Example usage:
dfa = PatientIDValidatorDFA()
print(dfa.validate("PT-12345"))  # True
print(dfa.validate("PX-12345"))  # False

True
False


Code 4: Finite-State Morphological Parser FST (Task 7)

In [3]:
class SimpleMorphologicalFST:
    def __init__(self):
        # Lexicon mapping surface forms to (stem, features)
        self.lexicon = {
            "fracture": ("fracture", ["+N", "+SG"]),
            "fractures": ("fracture", ["+N", "+PL"]),
            "fractured": ("fracture", ["+V", "+PAST"]),
            "fracturing": ("fracture", ["+V", "+PROG"]),
            "nephrectomy": ("nephrectomy", ["+N", "+SG"]),
            "nephrectomies": ("nephrectomy", ["+N", "+PL"])
        }

    def parse(self, surface_form):
        surface_lower = surface_form.lower()
        if surface_lower in self.lexicon:
            stem, features = self.lexicon[surface_lower]
            return f"{surface_form} -> {stem} " + " ".join(features)
        return f"{surface_form} -> UNKNOWN"

    def generate(self, stem, features):
        target_features = set(features)
        for surface, (s, feats) in self.lexicon.items():
            if s == stem and set(feats) == target_features:
                return surface
        return None

# Example usage:
fst = SimpleMorphologicalFST()
print(fst.parse("fractures"))
print(fst.generate("fracture", ["+N", "+PL"]))

fractures -> fracture +N +PL
fractures


Code 5: Porter Stemmer Integration (Task 9)

In [2]:
from nltk.stem import PorterStemmer

def run_porter_stemmer(terms):
    ps = PorterStemmer()
    results = {}
    for term in terms:
        results[term] = ps.stem(term)
    return results

# Example clinical search terms:
clinical_terms = ["fracture", "fractures", "fractured", "fracturing", "nephrectomy", "nephrectomies"]
stemmed_results = run_porter_stemmer(clinical_terms)

for term, stem in stemmed_results.items():
    print(f"Surface: {term} ---> Stem: {stem}")

Surface: fracture ---> Stem: fractur
Surface: fractures ---> Stem: fractur
Surface: fractured ---> Stem: fractur
Surface: fracturing ---> Stem: fractur
Surface: nephrectomy ---> Stem: nephrectomi
Surface: nephrectomies ---> Stem: nephrectomi
